# Heterogeneous mixing introduction

Adapted from the [Monash EMU summer textbook](https://github.com/monash-emu/summer-textbook)
notebook `textbook/12-heterogeneous-mixing-intro.ipynb` at commit
`fd97783474789e50ace5ea420aec20147f9bbd76`.

Source licence: BSD-2-Clause, Copyright (c) 2022, monash-emu. Prose is carried
and adapted; code is written in summer4 idiom.

Heterogeneous mixing between population sub-groups is one of the harder parts
of compartmental epidemic modelling. This chapter introduces what it is, why
we might want it, and the matrix notation summer4 uses. Later chapters build
on the same `MixingMatrix` / `ForceOfInfection` surface
({doc}`14-assortative-mixing`, {doc}`15-susceptibility-infectiousness-matrices`).
Chapter 13 continues with frequency vs density contact interpretation of
matrix entries and population splits — see {doc}`13-mixing-and-transmission-types`.



## Rationale

Direct transmission is driven by interactions between people. Models so far
have assumed **homogeneous mixing**: any two individuals have the same chance
of effective contact. That is convenient (like particles in Brownian motion)
but often wrong for humans.

One alternative is agent-based or network modelling. Staying compartmental,
we stratify the population and allow a **rate of interaction between every
pair of strata** — including a stratum with itself. With $n$ strata that is
$n^{2}$ interaction types. Multiple stratifications can stack, but we start
with the simplest two-group case.

## Assumptions and conventions

- Infection is directly transmitted.
- Stratifications are complete and mutually exclusive (if there is a
  `rural` group there is also an `urban` group covering the rest).
- Matrix rows and columns follow the same trait order: cell $(0, 0)$ is
  contact of the first group with itself.

In summer4, disease states and mixing groups are both
{class}`~summer4.Property` axes on a {class}`~summer4.PropertyMap`.



In [ ]:
import numpy as np
import pandas as pd
import plotly.io as pio

from summer4 import (
    Compartments,
    GroupedOutput,
    Property,
    PropertyMap,
    SavePlan,
    SaveRequest,
    FlowModel,
    TransitionFlow,
)
from summer4.epi import ForceOfInfection, MixingMatrix

pd.options.plotting.backend = "plotly"
pio.renderers.default = "notebook_connected"

state = Property("state", ("susceptible", "infectious"))
group = Property("group", ("group1", "group2"))
pmap = PropertyMap.from_property(state).stratify(group)

assert pmap.size == 4
assert list(group.traits) == ["group1", "group2"]
print("compartments:", [pmap.label(i) for i in range(pmap.size)])


## Heterogeneous mixing matrix

Each of two subgroups can interact with either subgroup, including itself.
summer4 takes a square array aligned to the grouping property's traits.

**Convention (same as summer2):** rows are the population being *infected*,
columns are the population doing the *infecting*. If $\beta_{i,j}$ is a
matrix entry, $i$ is the susceptible (infectee) group and $j$ is the
infectious (infector) group.

| | group1 (infector) | group2 (infector) |
|---|---|---|
| **group1** (infected) | $\beta_{1,1}$ | $\beta_{1,2}$ |
| **group2** (infected) | $\beta_{2,1}$ | $\beta_{2,2}$ |

Attach the matrix with {class}`~summer4.epi.MixingMatrix`. For density-
dependent contact rates interpreted as absolute per-capita×per-capita
weights, use `normalize="none"` so row sums are not forced to one.



In [ ]:
END_TIME = 40.0
PLAN = SavePlan(
    requests={
        "comp": SaveRequest(Compartments()),
        "foi": SaveRequest(GroupedOutput("infection")),
    },
    ts=np.linspace(0.0, END_TIME, int(END_TIME * 5) + 1),
)

# Homogeneous density weights: every group-pair contributes equally.
K = np.array(
    [
        [1.0, 1.0],  # group1 infected by group1, group2
        [1.0, 1.0],  # group2 infected by group1, group2
    ]
)

model = FlowModel(pmap)

mixing = MixingMatrix(group, K, normalize="none", check_reciprocal=False)
model.add_flow(
    TransitionFlow(
        "infection",
        state["susceptible"],
        state["infectious"],
        ForceOfInfection(
    "infection",
    infectious=state["infectious"],
    group_by=group,
    kind="density",
    contact_rate=1.0,
    mixing=mixing,
),
    )
)
# SI snapshot: no recovery flow — we only need λ(t) under the mixing matrix.

y0 = np.zeros(pmap.size)
y0[pmap.select(state["susceptible"] & group["group1"])] = 0.5
y0[pmap.select(state["susceptible"] & group["group2"])] = 0.5
y0[pmap.select(state["infectious"] & group["group1"])] = 0.01
y0[pmap.select(state["infectious"] & group["group2"])] = 0.01

res = model.compile().run(
    {}, y0, t0=0.0, t1=END_TIME, dt=0.1, save=PLAN, solver="euler"
)

foi = res["foi"].to_pandas()
foi.columns = list(group.traits)
assert res["foi"].dims == ("time", "group")
# Ones matrix + equal infectious seeds ⇒ identical λ in both groups.
assert float(np.max(np.abs(foi["group1"] - foi["group2"]))) < 1e-12

foi.plot(
    title="Force of infection by group (homogeneous density matrix)",
    labels={"index": "time", "value": "λ (per day)"},
)


## Equivalent equations

Under density dependence with contact rate absorbed into the matrix (or set
to 1), the force of infection in `group1` is

$$
\lambda_1(t) = \beta_{1,1}\,I_1(t) + \beta_{1,2}\,I_2(t).
$$

So $\lambda_1$ has a within-group piece and a between-group piece. The
equation for $\lambda_2$ is the second row of the same product
$\boldsymbol{\lambda} = K\,\mathbf{I}$. summer4 evaluates that product inside
{class}`~summer4.epi.ForceOfInfection`; saving
{class}`~summer4.GroupedOutput` `"infection"` exposes $\lambda$ as a
`(time, group)` trace without hand-slicing compartments.

